This is part of the pipeline I used to generate the following dataset with 3k+ essays

https://www.kaggle.com/datasets/illidan7/pii-detect-illi-train-dataset

_____

- Explore the competition dataset to better understand the structure of the essays and the PIIs in the data

https://www.kaggle.com/code/illidan7/pii-detect-data-explore

- Generate PIIs; Generate PII targets that closely resembled the ones in the competition data

https://www.kaggle.com/code/illidan7/pii-detect-mistral-pii-generation/notebook
https://www.kaggle.com/datasets/illidan7/pii-detect-generated-piis

- Generate Essays; Generate essays similar in structure to the competition data

https://www.kaggle.com/code/illidan7/pii-detect-mistral-dataset-generation

# Install packages

In [ ]:
%%time
from IPython.display import clear_output

!pip install faker

! pip install -q -U transformers
! pip install -q -U accelerate
! pip install -q -U bitsandbytes

! pip install -qq -U langchain

clear_output()

# Load libraries

In [ ]:
%%time

import sys, random, string, re, time, os
import warnings
warnings.filterwarnings("ignore")
import gc
import time
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch

### transformers
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

### quantization
import bitsandbytes as bnb

### langchain
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain import PromptTemplate, LLMChain
from langchain.llms import HuggingFacePipeline
import langchain

from faker import Faker  #generates fake data 
from spacy.lang.en import English

In [ ]:
import torch
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {DEVICE}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Pytorch {torch.__version__}")

In [ ]:
import torch, random
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SEED = 42
# Seed the same seed to all 
def seed_everything(seed=42):
    Faker.seed(0)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

In [ ]:
import ctypes, gc, torch
libc = ctypes.CDLL("libc.so.6")
def clear_memory():
    libc.malloc_trim(0)
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
print('torch version: ', torch.__version__)
print(f'transformers version: {transformers.__version__}')
print(f'bnb version: {bnb.__version__}')
print(f'langchain version: {langchain.__version__}')

# Configs

In [ ]:
class CFG:
    
    ### model
    MODEL_PATH = '/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1'    
    

# Load LLM `Mistral-7b-instruct-v0.1-hf` 


In [ ]:
def load_model():
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)
    
    ### quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.float16,
        bnb_4bit_use_double_quant = True,
        llm_int8_enable_fp32_cpu_offload = True,
    )
#     tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    
    ### tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        CFG.MODEL_PATH,
        trust_remote_code = True, 
        use_fast=True
    )
    
    ### model
    model = AutoModelForCausalLM.from_pretrained(
        CFG.MODEL_PATH,
        quantization_config = bnb_config,
#         torch_dtype=torch.bfloat16,
        device_map = "auto",
        trust_remote_code = True,
    #     attn_implementation = 'flash_attention_2',
    )
    
    return model, tokenizer

In [ ]:
%%time

model, tokenizer = load_model()

# Read in Generated PII

In [ ]:
pii_df = pd.read_csv("/kaggle/input/pii-detect-generated-piis/pii_mistral.csv")

pii_df.shape

In [ ]:
pii_df.pii_type.value_counts()

In [ ]:
pii_data = {pii_type: pii_df[pii_df['pii_type'] == pii_type]['pii_id'].tolist() for pii_type in pii_df['pii_type'].unique()}

pii_data.keys()

In [ ]:
pii_types_distribution = {
                                        'NAME_STUDENT': 0.7,
                                        'URL_PERSONAL': 0.1,
                                        'ID_NUM': 0.05,
                                        'EMAIL': 0.05,
                                        'PHONE_NUM': 0.05,
                                        'STREET_ADDRESS': 0.05
                                    }

# Generate essay function

In [ ]:
def generate_essay(length, num_pii=0, pii_df=None):
    topic = np.random.choice(['Learning Launch', 'Visualization', 'Storytelling', 'Visual Thinking', 'Mind mapping', 'Design thinking'])
    
    if pii_df is not None: 
    
        # Determine PII types to include in the essay
        pii_selection = np.random.choice(list(pii_types_distribution.keys()), 
                                         size=num_pii, 
                                         replace=True, 
                                         p=list(pii_types_distribution.values()))

        # Sample PIIs from the pii_df based on pii_selection
        pii_details = ""
        for pii_type in pii_selection:
            pii_sample = pii_df[pii_df['pii_type'] == pii_type]['pii_id'].sample(1).values[0]
            pii_details += f"{pii_type}: {pii_sample}\n"

        prompt_template = f"""<s>[INST]
        You are a student enrolled in a massively open online course (MOOC) and are tasked with writing an essay on "{topic}" to apply the course material to address a real-world problem. As part of your essay, you decide to include a personal story or example that directly involves you or someone you know, which includes the following personal information (PII) to enrich the narrative:

        {pii_details}

        In your essay, make sure to creatively integrate this personal information in a way that enhances your argument or story. You might use the STREET_ADDRESS to set the scene, the URL_PERSONAL as a reference to an online portfolio or project, and the NAME_STUDENT to personalize the story, for example.

        The essay should include some or all of the following sections:

        [CHALLENGES]
        Here, describe the challenges you (or the person in your example) faced when applying the course material to the real-world problem.

        [SELECTION]
        Elaborate on why certain solutions were selected to overcome these challenges.

        [INSIGHT]
        Share any insights gained from confronting these challenges and implementing the solutions.

        [APPLICATION]
        Detail how these insights were applied to a real-world scenario, making sure to highlight the impact of integrating the personal information provided.

        [APPROACH]
        Conclude with an overview of the overall approach taken to weave together course material with practical, real-world solutions, emphasizing how the included PII helped in forming a more compelling narrative or argument.

        Remember, the goal is to demonstrate how visual thinking, coupled with real-life applications and personal examples, can lead to innovative solutions and insights.
        [/INST]"""    

        # Fill in prompt with PII
        prompt = prompt_template.format(topic=topic,
                                        pii_details=pii_details
                                       )
    
    else:
        
        prompt_template = f"""<s>[INST]
        You are a student enrolled in a massively open online course (MOOC) and are tasked with writing an essay on "{topic}" to apply the course material to address a real-world problem. 
        
        The essay should include some or all of the following sections:

        [CHALLENGES]
        Here, describe the challenges you (or the person in your example) faced when applying the course material to the real-world problem.

        [SELECTION]
        Elaborate on why certain solutions were selected to overcome these challenges.

        [INSIGHT]
        Share any insights gained from confronting these challenges and implementing the solutions.

        [APPLICATION]
        Detail how these insights were applied to a real-world scenario, making sure to highlight the impact of integrating the personal information provided.

        [APPROACH]
        Conclude with an overview of the overall approach taken to weave together course material with practical, real-world solutions, emphasizing how the included PII helped in forming a more compelling narrative or argument.

        Remember, the goal is to demonstrate how visual thinking, coupled with real-life applications and personal examples, can lead to innovative solutions and insights.
        [/INST]"""    

        # Fill in prompt with PII
        prompt = prompt_template.format(topic=topic
                                       )
        

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=length,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    if pii_df is None:
        return generated_text, ''
    
    return generated_text, pii_details

# Generate essays

In [ ]:
%%time

# Distribution parameters 
avg_length = 733
std_length = 319
min_length = 69
max_length = 3298

total_essays = 325
proportion_without_pii = 0.3

# Generate essays
lengths = []
include = []
numpii = []
essays = []
piis = []

for _ in tqdm(range(total_essays)):
    # Determine essay length based on the provided distribution
    length = max(min_length, min(int(np.random.normal(avg_length, std_length)), max_length))
    
    # Decide whether to include PII
    include_pii = np.random.rand() > proportion_without_pii
    num_pii = 0
    if include_pii:
        # For simplicity, assume a uniform distribution of 1-6 PIIs
        num_pii = np.random.randint(1, 7)
    
    # Generate the essay
    essay, pii = generate_essay(length=length, num_pii=num_pii, pii_df=None)
    
    # If PIIs need to be included, insert them
    if num_pii > 0:
        essay, pii = generate_essay(length=length, num_pii=num_pii, pii_df=pii_df)
    
    lengths.append(length)
    include.append(include_pii)
    numpii.append(num_pii)
    essays.append(essay)
    piis.append(pii)


# Creating a DataFrame
df = pd.DataFrame({'lengths': lengths,
                   'include': include,
                   'numpii': numpii,
                   'essays': essays,
                   'piis': piis})

# Convert to Competition format

In [ ]:
from spacy.lang.en import English
en_tokenizer = English().tokenizer

def tokenize_with_spacy(text, tokenizer=en_tokenizer):
    tokenized_text = tokenizer(text)
    tokens = [token.text for token in tokenized_text]
    trailing_whitespace = [bool(token.whitespace_) for token in tokenized_text]
    return tokens, trailing_whitespace

In [ ]:
toa_data = []

start_document_id = 1000000

for index, row in tqdm(enumerate(toa_df.itertuples()), total=len(toa_df)):
    
    full_text = getattr(row,'essays').replace('\r\n', '\n')
    
    tokens, trailing_whitespace = tokenize_with_spacy(full_text, en_tokenizer)
    
    labels = ['O'] * len(tokens)
    
    if getattr(row,'numpii') > 0:
        
        pii_dict = {f"{pii.split(': ')[0].strip()}_{i}": pii.split(": ")[1].strip() for i, pii in enumerate(getattr(row,'piis').split("\n")) if len(pii)>10 and ': ' in pii}

        for pii_type, pii_id in pii_dict.items():

            pii_id_ = pii_id.split()

            for i, token in enumerate(tokens):
                if token in pii_id_:

                    labels[i] = f"B-{pii_type[:-2]}" 

                    if labels[i-1] == f"B-{pii_type[:-2]}":
                        labels[i] = f"I-{pii_type[:-2]}"
    
    
    toa_data.append({
                    "document": start_document_id + index,
                    "full_text": full_text,
                    "tokens": tokens,
                    "trailing_whitespace": trailing_whitespace,
                    "labels": labels
                    })

# Save to json

In [ ]:
# Path where the JSON file will be saved
file_path = './new_dataset.json'

# Saving the list of dictionaries to a JSON file
import json

with open(file_path, 'w') as file:
    json.dump(toa_data, file, indent=4)